# HMM Noun Phrase Chunking (B-NP / I-NP / O)
1. Train an HMM (transition + emission counts, Add-K smoothing)
2. **Forward algorithm** → probability of each test sentence
3. **Viterbi decoding** → predicted NP tags
4. Evaluate with `seqeval`

In [6]:
!pip install -q nltk
!pip install -q setuptools wheel
!pip install -q seqeval --no-build-isolation || echo 'seqeval install failed - built-in fallback evaluator will be used'

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
seqeval install failed - built-in fallback evaluator will be used


In [7]:
import numpy as np, pandas as pd
from collections import Counter
from scipy.special import logsumexp
from sklearn.model_selection import train_test_split

try:
    from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
    print("using seqeval")
except Exception:
    print("seqeval unavailable -> using built-in chunk evaluator (same BIO span-matching logic)")

    def _spans(seq):
        spans, start, typ = set(), None, None
        for i, tag in enumerate(list(seq) + ["O"]):
            if tag == "O" or tag.startswith("B-") or (tag.startswith("I-") and tag[2:] != typ):
                if start is not None: spans.add((typ, start, i))
                start, typ = (i, tag[2:]) if tag != "O" else (None, None)
        return spans

    def _counts(y_true, y_pred):
        tp = fp = fn = 0
        for g, p in zip(y_true, y_pred):
            G, P = _spans(g), _spans(p)
            tp += len(G & P); fp += len(P - G); fn += len(G - P)
        return tp, fp, fn

    def precision_score(y_true, y_pred):
        tp, fp, _ = _counts(y_true, y_pred); return tp / (tp + fp) if tp + fp else 0.0
    def recall_score(y_true, y_pred):
        tp, _, fn = _counts(y_true, y_pred); return tp / (tp + fn) if tp + fn else 0.0
    def f1_score(y_true, y_pred):
        p, r = precision_score(y_true, y_pred), recall_score(y_true, y_pred)
        return 2 * p * r / (p + r) if p + r else 0.0
    def classification_report(y_true, y_pred, digits=4):
        p, r, f = precision_score(y_true, y_pred), recall_score(y_true, y_pred), f1_score(y_true, y_pred)
        tp, fp, fn = _counts(y_true, y_pred)
        return (f"{'':>10}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}\n"
                f"{'NP':>10}{p:>10.{digits}f}{r:>10.{digits}f}{f:>10.{digits}f}{tp+fn:>10}\n")

seqeval unavailable -> using built-in chunk evaluator (same BIO span-matching logic)


## 1. Load data
Set `TRAIN_PATH` / `TEST_PATH` to your files (CoNLL style: one token per line, **first column = word, last column = tag**, blank line between sentences). If left as `None`, the notebook falls back to NLTK's CoNLL-2000 (non-NP chunks mapped to `O`) so you can test the pipeline.

In [8]:
TRAIN_PATH = None   # e.g. "/kaggle/input/your-data/train.txt"
TEST_PATH  = None   # e.g. "/kaggle/input/your-data/test.txt"
LOWERCASE  = True

def read_conll(path):
    sents, cur = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if cur: sents.append(cur); cur = []
                continue
            p = line.split()
            cur.append((p[0], p[-1]))
    if cur: sents.append(cur)
    return sents

def load_conll2000():
    import nltk
    nltk.download("conll2000", quiet=True)
    from nltk.corpus import conll2000
    conv = lambda ss: [[(w, c if c.endswith("NP") else "O") for w, _, c in s] for s in ss]
    return conv(conll2000.iob_sents("train.txt")), conv(conll2000.iob_sents("test.txt"))

if TRAIN_PATH and TEST_PATH:
    train_data, test_data = read_conll(TRAIN_PATH), read_conll(TEST_PATH)
else:
    train_data, test_data = load_conll2000()

norm = (lambda w: w.lower()) if LOWERCASE else (lambda w: w)
train_data = [[(norm(w), t) for w, t in s] for s in train_data]
test_data  = [[(norm(w), t) for w, t in s] for s in test_data]
print(len(train_data), "train sents |", len(test_data), "test sents")
print(train_data[0])

8936 train sents | 2012 test sents
[('confidence', 'B-NP'), ('in', 'O'), ('the', 'B-NP'), ('pound', 'I-NP'), ('is', 'O'), ('widely', 'O'), ('expected', 'O'), ('to', 'O'), ('take', 'O'), ('another', 'B-NP'), ('sharp', 'I-NP'), ('dive', 'I-NP'), ('if', 'O'), ('trade', 'B-NP'), ('figures', 'I-NP'), ('for', 'O'), ('september', 'B-NP'), (',', 'O'), ('due', 'O'), ('for', 'O'), ('release', 'B-NP'), ('tomorrow', 'B-NP'), (',', 'O'), ('fail', 'O'), ('to', 'O'), ('show', 'O'), ('a', 'B-NP'), ('substantial', 'I-NP'), ('improvement', 'I-NP'), ('from', 'O'), ('july', 'B-NP'), ('and', 'I-NP'), ('august', 'I-NP'), ("'s", 'B-NP'), ('near-record', 'I-NP'), ('deficits', 'I-NP'), ('.', 'O')]


## 2. HMM with Add-K smoothing
- Start: `P(t1) = (C(<s>→t1)+k) / (N+k·T)`
- Transition: `P(tj|ti) = (C(ti→tj)+k) / (C(ti)+k·T)`
- Emission: `P(w|t) = (C(t,w)+k) / (C(t)+k·(V+1))` — the `+1` is an `<UNK>` slot, so unseen test words get a non-zero probability.

Everything is computed in **log space** to avoid underflow.

In [9]:
class HMM:
    UNK = "<UNK>"

    def __init__(self, k=0.01):
        self.k = k

    def fit(self, data):
        k = self.k
        self.tags = sorted({t for s in data for _, t in s})
        self.t2i = {t: i for i, t in enumerate(self.tags)}
        vocab = sorted({w for s in data for w, _ in s})
        self.w2i = {w: i for i, w in enumerate(vocab)}
        self.w2i[self.UNK] = len(self.w2i)
        T, V = len(self.tags), len(self.w2i)

        start, trans, emit = np.zeros(T), np.zeros((T, T)), np.zeros((T, V))
        for s in data:
            prev = None
            for w, t in s:
                ti = self.t2i[t]
                emit[ti, self.w2i[w]] += 1
                if prev is None: start[ti] += 1
                else: trans[prev, ti] += 1
                prev = ti

        self.log_pi = np.log((start + k) / (start.sum() + k * T))
        self.log_A  = np.log((trans + k) / (trans.sum(1, keepdims=True) + k * T))
        self.log_B  = np.log((emit + k) / (emit.sum(1, keepdims=True) + k * V))
        return self

    def encode(self, words):
        unk = self.w2i[self.UNK]
        return [self.w2i.get(w, unk) for w in words]

    # ---- Forward algorithm: log P(O) = log sum over all tag paths ----
    def forward(self, words):
        o = self.encode(words)
        alpha = self.log_pi + self.log_B[:, o[0]]
        for t in range(1, len(o)):
            alpha = logsumexp(alpha[:, None] + self.log_A, axis=0) + self.log_B[:, o[t]]
        return logsumexp(alpha)

    # ---- Viterbi: most likely tag sequence ----
    def viterbi(self, words):
        o = self.encode(words)
        n, T = len(o), len(self.tags)
        delta = np.zeros((n, T)); back = np.zeros((n, T), dtype=int)
        delta[0] = self.log_pi + self.log_B[:, o[0]]
        for t in range(1, n):
            scores = delta[t-1][:, None] + self.log_A      # (prev, cur)
            back[t] = scores.argmax(0)
            delta[t] = scores.max(0) + self.log_B[:, o[t]]
        path = [int(delta[-1].argmax())]
        for t in range(n - 1, 0, -1):
            path.append(int(back[t, path[-1]]))
        path.reverse()
        return [self.tags[i] for i in path], delta[-1].max()

## 3. Tune K on a dev split

In [10]:
def evaluate(model, data):
    y_true, y_pred = [], []
    for s in data:
        words, gold = [w for w, _ in s], [t for _, t in s]
        pred, _ = model.viterbi(words)
        y_true.append(gold); y_pred.append(pred)
    return y_true, y_pred

tr, dev = train_test_split(train_data, test_size=0.1, random_state=42)
results = []
for k in [1e-5, 1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.5, 1.0]:
    m = HMM(k).fit(tr)
    yt, yp = evaluate(m, dev)
    results.append((k, f1_score(yt, yp)))
    print(f"k={k:<8} dev F1={results[-1][1]:.4f}")
best_k = max(results, key=lambda x: x[1])[0]
print("Best k:", best_k)

k=1e-05    dev F1=0.8077
k=0.0001   dev F1=0.8077
k=0.001    dev F1=0.8094
k=0.01     dev F1=0.8134
k=0.05     dev F1=0.8191
k=0.1      dev F1=0.8238
k=0.5      dev F1=0.8194
k=1.0      dev F1=0.8168
Best k: 0.1


## 4. Train on full training data, then run Forward + Viterbi on test set

In [11]:
model = HMM(best_k).fit(train_data)

rows = []
y_true, y_pred = [], []
for i, s in enumerate(test_data):
    words, gold = [w for w, _ in s], [t for _, t in s]
    logp = model.forward(words)                 # task 1
    pred, vit_logp = model.viterbi(words)       # task 2
    assert vit_logp <= logp + 1e-9              # best path can't exceed total prob
    y_true.append(gold); y_pred.append(pred)
    rows.append({"sent_id": i, "sentence": " ".join(words),
                 "log_prob_forward": logp, "prob_forward": float(np.exp(logp)),
                 "log_prob_viterbi": vit_logp,
                 "gold": " ".join(gold), "pred": " ".join(pred)})

df = pd.DataFrame(rows)
df.to_csv("hmm_np_predictions.csv", index=False)
df[["sent_id", "sentence", "log_prob_forward", "prob_forward"]].head(10)

,sent_id,sentence,log_prob_forward,prob_forward
0,0,rockwell international corp. 's tulsa unit sai...,-222.262044,2.970440e-97
1,1,rockwell said the agreement calls for it to su...,-120.829056,3.346629e-53
2,2,"these include , among other parts , each jetli...",-253.215289,1.071512e-110
3,3,"under the existing contract , rockwell said , ...",-132.394790,3.174486e-58
4,4,"rockwell , based in el segundo , calif. , is a...",-133.925171,6.871275e-59
5,5,frank carlucci iii was named to this telecommu...,-181.161861,2.100891e-79
6,6,"mr. carlucci , 59 years old , served as defens...",-107.596348,1.868529e-47
7,7,"in january , he accepted the position of vice ...",-121.452855,1.793475e-53
8,8,shearson lehman hutton inc .,-37.113818,7.615062e-17
9,9,"thomas e. meador , 42 years old , was named pr...",-227.648189,1.360341e-99


## 5. Evaluation (seqeval)

In [12]:
print(classification_report(y_true, y_pred, digits=4))
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
print(f"F1:        {f1_score(y_true, y_pred):.4f}")

           precision    recall  f1-score   support
        NP    0.7990    0.8280    0.8133     12422

Precision: 0.7990
Recall:    0.8280
F1:        0.8133


## 6. Example prediction

In [13]:
ex = test_data[0]
words = [w for w, _ in ex]
pred, _ = model.viterbi(words)
print(pd.DataFrame({"word": words, "gold": [t for _, t in ex], "pred": pred}).T.to_string(header=False))
print("\nForward log P(sentence):", model.forward(words))

word  rockwell  international  corp.    's  tulsa  unit  said    it  signed     a  tentative  agreement  extending   its  contract  with  boeing   co.  to  provide  structural  parts  for  boeing    's   747  jetliners  .
gold      B-NP           I-NP   I-NP  B-NP   I-NP  I-NP     O  B-NP       O  B-NP       I-NP       I-NP          O  B-NP      I-NP     O    B-NP  I-NP   O        O        B-NP   I-NP    O    B-NP  B-NP  I-NP       I-NP  O
pred      B-NP           I-NP   I-NP  B-NP   I-NP  I-NP     O  B-NP       O  B-NP       I-NP       I-NP          O  B-NP      I-NP     O    B-NP  I-NP   O        O        B-NP   I-NP    O    B-NP     O  B-NP       I-NP  O

Forward log P(sentence): -222.26204383097058
